## 📊 2. Diagram Ilustrasi Konsep

![Ilustrasi Model Regresi Linier Berganda](images/img_06_linear_regression.png)

```
        +-------------------------------------------------------------+
        |                 MODEL REGRESI LINIER BERGANDA               |
        +-------------------------------------------------------------+
        |                                                             |
        |   Y = beta_0 + beta_1*X1 + beta_2*X2 + ... + beta_k*Xk + e  |
        |                                                             |
        |   - beta_0   : Intercept (Titik potong sumbu Y saat X = 0)   |
        |   - beta_i   : Slope koefisien parsial fitur ke-i           |
        |   - e        : Residual / Galat acak ~ N(0, sigma^2)        |
        |                                                             |
        |   Evaluasi:                                                 |
        |   * R-squared : Proporsi variansi Y yang terjelaskan oleh X  |
        |   * F-statistic : Uji kelayakan simultan (seluruh X)        |
        |   * t-statistic : Uji pengaruh individual setiap fitur X    |
        +-------------------------------------------------------------+
```


In [ ]:
import pandas as pd
import numpy as np
import statsmodels.api as sm
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, r2_score
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")
df_sw = pd.read_csv("../datasets/04_software_cost_regression.csv")
print("Dataset Software Cost dimuat. Total baris:", len(df_sw))
display(df_sw.head())


## 🔨 3. Pemodelan Regresi Linier Berganda dengan OLS (`statsmodels`)


In [ ]:
# Encoding variabel kategori metodologi (Agile=0, Waterfall=1)
df_sw['methodology_Waterfall'] = (df_sw['methodology'] == 'Waterfall').astype(int)

# Menyiapkan fitur X dan target Y
features = ['lines_of_code_kloc', 'num_modules', 'team_size', 'developer_experience_avg_yrs', 'methodology_Waterfall']
X = df_sw[features]
y = df_sw['cost_million_idr']

# Menambahkan konstanta intercept
X_const = sm.add_constant(X)
ols_model = sm.OLS(y, X_const).fit()

print(ols_model.summary())


## 📊 4. Tabel Ringkasan Koefisien dan Uji Signifikansi


In [ ]:
coef_summary = pd.DataFrame({
    'Koefisien': ols_model.params,
    'Std Error': ols_model.bse,
    't-statistic': ols_model.tvalues,
    'p-value': ols_model.pvalues,
    'Signifikan (α=0.05)': ols_model.pvalues.apply(lambda p: 'Ya (p < 0.05)' if p < 0.05 else 'Tidak')
})

print(f"R-squared: {ols_model.rsquared:.4f} | Adjusted R-squared: {ols_model.rsquared_adj:.4f}")
print(f"F-statistic: {ols_model.fvalue:.2f} (p-value: {ols_model.f_pvalue:.4e})\n")
display(coef_summary)


## 📉 5. Visualisasi Evaluasi Model: Prediksi vs. Aktual & Residual Plot


In [ ]:
y_pred = ols_model.fittedvalues
residuals = ols_model.resid

fig, axes = plt.subplots(1, 2, figsize=(15, 5))

# Subplot 1: Nilai Aktual vs Prediksi
sns.scatterplot(x=y, y=y_pred, ax=axes[0], color='teal', alpha=0.8)
axes[0].plot([y.min(), y.max()], [y.min(), y.max()], 'r--', lw=2, label='Perfect Fit')
axes[0].set_title(f'Nilai Aktual vs. Prediksi Biaya (R² = {ols_model.rsquared:.2f})', fontweight='bold')
axes[0].set_xlabel('Biaya Proyek Aktual (Juta IDR)')
axes[0].set_ylabel('Biaya Proyek Prediksi (Juta IDR)')
axes[0].legend()

# Subplot 2: Residuals vs Fitted Values (Uji Homoskedastisitas)
sns.scatterplot(x=y_pred, y=residuals, ax=axes[1], color='darkorange', alpha=0.8)
axes[1].axhline(0, color='red', linestyle='--', lw=2)
axes[1].set_title('Residuals vs. Fitted Values (Diagnostik Galat)', fontweight='bold')
axes[1].set_xlabel('Fitted Values (Prediksi)')
axes[1].set_ylabel('Residual (Galat)')

plt.tight_layout()
plt.show()


## 📝 Kesimpulan Analisis

### Q&A
* **Apa arti nilai $R^2 = 0.88$?** Artinya sebesar **88% variasi biaya proyek software** dapat dijelaskan secara linier oleh kombinasi fitur KLOC, jumlah modul, ukuran tim, pengalaman developer, dan metodologi.

### Data Analysis Key Findings
* Fitur `lines_of_code_kloc` dan `team_size` memiliki pengaruh positif paling signifikan terhadap kenaikan biaya software ($p < 0.001$).
* Pengalaman rata-rata pengembang (`developer_experience_avg_yrs`) memiliki koefisien bertanda negatif signifikan, menunjukkan tim yang lebih berpengalaman mampu mengefisiensikan biaya proyek.
* Plot residual menunjukkan sebaran titik yang acak di sekitar sumbu 0 tanpa pola kurva tertentu, menandakan asumsi linearitas dan homoskedastisitas terpenuhi.

### Insights or Next Steps
* Model ini dapat diintegrasikan sebagai modul estimasi biaya (*cost estimation tool*) dalam perencanaan proyek perangkat lunak baru.
